In [ ]:

from openai import OpenAI
import gradio as gr
import os
import json
import logging
from typing import Any
from pydantic import BaseModel
from e2b_code_interpreter import Sandbox
from handle_tool_calls import handle_tool_calls
from summariser import call_summariser
from dotenv import load_dotenv
from chromadb import Client
import uuid
from sentence_transformers import SentenceTransformer

load_dotenv()


In [ ]:
api_key = os.getenv("NVIDIA_API_KEY")

url = os.getenv("URL")
print("url:", url)

SUMMARY_MODEL="bytedance/seed-oss-36b-instruct"
CHAT_MODEL="moonshotai/kimi-k2-instruct-0905"


In [ ]:
class Message(BaseModel):
    role: str
    content: str

In [ ]:
client = OpenAI(
  base_url = url,
  api_key = api_key
)

In [ ]:
system = "You are a helpful assistant"

In [ ]:
total_tokens = 0

code_n_execute_JSON = {
    "type": "function",
    "function": {
        "name": "code_n_execute",
        "description": "Generates Python code using an LLM based on a prompt, then executes it in an E2B sandbox and returns the result.",
        "parameters": {
            "type": "object",
            "properties": {
                "prompt": {
                    "type": "string",
                    "description": "The task or question that requires Python code to solve.",
                    "default": "Calculate how many r's are in the word 'strawberrrry'",
                }
            },
            "required": ["prompt"],
        },
    },
}

google_search_JSON = {
    "type": "function",
    "function": {
        "name": "google_search",
        "description": "Search Google using SerpAPI and return results for a query",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query to look up on Google"
                }
            },
            "required": ["query"],
        },
    },
}

tools = [code_n_execute_JSON, google_search_JSON]

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    global total_tokens
    context = "\n".join(retrieve_from_RAG(message))
    print("Retrieved context:", context)
    system_with_context = f"{system}\n\nRelevant past context:\n{context}"

    messages = [{"role": "system", "content": system_with_context}] + history + [{"role": "user", "content": message}]

    # if total_tokens > 10000:
    #     messages = call_summariser(history)
    
    response = client.chat.completions.create(model=CHAT_MODEL, messages=messages, tools=tools)
    total_tokens += response.usage.total_tokens
    print("Total tokens current:", total_tokens)


    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_calls(message)
        messages.append(message)
        for item in response:
            messages.append(item)
        response = client.chat.completions.create(model=CHAT_MODEL, messages=messages)
    
    save_to_RAG(message, response.choices[0].message.content)


    return response.choices[0].message.content  


In [ ]:
chroma_client = Client()
collection = chroma_client.get_or_create_collection(name="chat_memory")


EMBEDDING_MODEL = SentenceTransformer("BAAI/bge-small-en-v1.5")

def save_to_RAG(query, response):
    query_embedding = EMBEDDING_MODEL.encode([query])[0].tolist()
    response_embedding = EMBEDDING_MODEL.encode([response])[0].tolist()
    
    turn_id = str(uuid.uuid4())
    
    collection.add(
        documents=[query, response],
        embeddings=[query_embedding, response_embedding],
        ids=[f"query_{turn_id}", f"response_{turn_id}"]
    )


def retrieve_from_RAG(query, top_k=3):
    query_embedding = EMBEDDING_MODEL.encode([query])[0].tolist()
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    
    return results["documents"][0]  # list of top-k relevant messages

In [ ]:
gr.ChatInterface(chat).launch()